In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from typing import List, Dict

# 1. LOAD DATA
# Ensure the CSV is in the same directory
df = pd.read_csv('us_tornado_dataset_1950_2021.csv')

# --- CLEANING STEP ---
# Calculate how many rows have 'mag' equal to -9
unknown_count = df[df['mag'] == -9].shape[0]
total_rows = df.shape[0]

print(f"Total tornadoes loaded: {total_rows}")
print(f"Tornadoes with unknown magnitude (-9): {unknown_count}")

# Perform the cleaning (Keep only rows where mag is 0 or greater)
df = df[df['mag'] >= 0]

print(f"Rows remaining for training: {df.shape[0]}")
print(f"Dropped {total_rows - df.shape[0]} rows.")

# 2. FEATURE SELECTION
# wid = Width, len = Length, slat/slon = Location, mo = Month
feature_cols = ['wid', 'len', 'slat', 'slon', 'mo']
X = df[feature_cols]
y = df['mag']

# 3. SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. TRAIN
print("Training Random Forest Model...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 5. REPORT
y_pred = rf_model.predict(X_test)
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

# ==========================================
# 6. NEW: TESTING LOGIC (INFERENCE)
# ==========================================

# A. Define the Human-Readable Labels
# These map to 0, 1, 2, 3, 4, 5
classes = ('EF0 (Light)', 'EF1 (Moderate)', 'EF2 (Significant)', 
                      'EF3 (Severe)', 'EF4 (Devastating)', 'EF5 (Incredible)')

# B. Define the Prediction Function
def predict_tornado_damage(tornadoes: List[Dict]):
        """
        Accepts a list of dictionaries describing tornadoes,
        and returns a list of text predictions (EF Scale).
        """
        # Convert list of dicts to DataFrame
        inputs = pd.DataFrame(tornadoes)

        # SAFETY: Ensure columns are in the exact same order as training
        # If we don't do this, providing {'len': 10, 'wid': 50} might swap the numbers!
        inputs = inputs[feature_cols]

        # Run prediction
        predictions = rf_model.predict(inputs)

        # Map the numeric output (e.g., 4) to the string label (e.g., 'EF4 ...')
        return [classes[p] for p in predictions]

# C. Define Test Cases
# Case 1: A massive, long-track tornado (Similar to historic EF4/5s)
severe_case = {
    "wid": 2000,    # 2000 yards wide (huge)
    "len": 25.0,    # 25 miles long
    "slat": 35.0,   # Oklahoma latitude
    "slon": -97.0,  # Oklahoma longitude
    "mo": 5         # May (Spring)
}

# Case 2: A small "rope" tornado
weak_case = {
    "wid": 30,      # 30 yards wide
    "len": 0.2,     # 0.2 miles long
    "slat": 41.0,   # Further North
    "slon": -87.0,
    "mo": 9         # September (Fall)
}

# D. Run the Prediction
print("\n--- Live Testing Results ---")
results = predict_tornado_damage([severe_case, weak_case])

for i, res in enumerate(results):
    print(f"Test Case #{i+1}: Predicted {res}")

Total tornadoes loaded: 67558
Tornadoes with unknown magnitude (-9): 605
Rows remaining for training: 66953
Dropped 605 rows.
Training Random Forest Model...

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.70      0.78      0.74      6269
           1       0.52      0.54      0.53      4609
           2       0.38      0.27      0.31      1888
           3       0.33      0.18      0.23       502
           4       0.31      0.10      0.15       110
           5       0.50      0.08      0.13        13

    accuracy                           0.60     13391
   macro avg       0.46      0.32      0.35     13391
weighted avg       0.58      0.60      0.58     13391


--- Live Testing Results ---
Test Case #1: Predicted EF4 (Devastating)
Test Case #2: Predicted EF0 (Light)
